<a href="https://colab.research.google.com/github/Patrick190508/whaleshark-mafia-island/blob/main/notebooks/MVP_Daily_Prediction_Ningaloo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MVP — Daily whale shark encounter prediction, Ningaloo Reef
**Minimum viable project: one site, one model, one honest evaluation.**

Question: given today's conditions, will at least one whale shark be sighted at Ningaloo?

- Data: OBIS human sightings matched to daily Copernicus SST, chlorophyll-a and salinity (`daily_ningaloo.csv`), peak season April–June, active seasons only.
- Target: `y` = 1 if at least one sighting that day, 0 otherwise. Days without a record count as 0 (observation-effort limitation).
- Baseline: average sighting probability per week of the season, from the training seasons.
- Model: logistic regression (Veritas Scholars, Week 4). Polynomial terms (Week 3) and a small neural network (Week 6) come after the MVP.
- Validation: training / validation / test sets split by season (time), never at random.

## 1. Data

### 1.1 Load the daily series and build the season table
One row per day of the season: environmental values of that day, day of season, week, target.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/Patrick190508/whaleshark-mafia-island/main/data/daily_ningaloo.csv"
d = pd.read_csv(URL, parse_dates=["date"])

PEAK = [4, 5, 6]                      # Ningaloo season: April–June
d = d[d["month"].isin(PEAK)].copy()
d["sday"] = d.groupby("year").cumcount() + 1      # day of season 1..91
d["week"] = (d["sday"] - 1) // 7 + 1
d["y"] = (d["sightings"] > 0).astype(int)         # target

# active seasons only (>= 20 sightings), as in the EDA
tot = d.groupby("year")["sightings"].sum()
active = tot[tot >= 20].index
d = d[d["year"].isin(active)].dropna(subset=["sst", "chl", "sss"])

print("seasons:", sorted(active))
print("rows:", len(d), "| days with sighting:", d["y"].sum(), f"({d['y'].mean():.0%})")

### 1.2 Train / validation / test split by season
Earliest seasons → training; later seasons → validation (to tune); most recent seasons → test (used once at the end).

## 2. Baseline

### 2.1 Weekly sighting probability from the training seasons

### 2.2 Baseline score on the validation set
Accuracy and confusion matrix when predicting 'sighting' in weeks whose probability is above 0.5.

*Reading:*

## 3. Logistic regression

### 3.1 Features and scaling
Inputs: day of season, SST, chlorophyll-a, salinity (standardised).

### 3.2 Fit on training, probabilities on validation

### 3.3 Threshold tuning on the validation set
Confusion matrix for several thresholds; choose the one that balances empty trips (false positives) and missed good days (false negatives).

*Reading:*

### 3.4 Coefficients — which variable moves the probability

*Reading:*

## 4. Test (used once)

### 4.1 Baseline vs logistic regression on the test seasons

*Reading:*

## 5. Conclusions and next steps

*Does the model beat the calendar?*

Next steps:
1. Polynomial terms (SST²) — Week 3
2. Small neural network — Week 6
3. Add wind speed as a fast-changing variable
4. Repeat for Philippines and Tofo; cross-site test